### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [1]:
## Import necessary libraries
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

C:\Users\dhruv\AppData\Local\Temp\ipykernel_26056\3788000821.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
## Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)

## Create chunks
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [3]:
## Step 2: FAISS Vector Store with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

## Step 3: Create MMR Retirever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A857FCB0E0>, search_type='mmr', search_kwargs={'k': 3})

In [4]:
## Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")

llm = init_chat_model(model = "openai/gpt-oss-120b", model_provider = "groq", temperature = 0.4)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A8590F1A90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A8590F3230>, model_name='openai/gpt-oss-120b', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
## Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm = llm, prompt = prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain = document_chain)

## Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 **LangChain’s support for agents**

* **Agent framework** – LangChain provides a generic “agent” abstraction that lets an LLM act as a planner and decide which tool to invoke, and in what order, to accomplish a user‑defined task.  
* **Tool integration** – Agents can call built‑in tools (e.g., calculators, web‑search, vector‑store retrieval) or any custom function you expose via the `Tool` interface. This makes it easy to hook external APIs, databases, or bespoke services into the reasoning loop.  
* **Dynamic tool selection** – The LLM receives a prompt that lists available tools and their descriptions; it then generates a “tool‑call” action, the system executes the call, returns the result, and the LLM continues reasoning. This iterative “think‑act‑observe” cycle enables complex, multi‑step workflows.  
* **Agent types** – LangChain ships ready‑made agents such as `ZeroShotAgent`, `ConversationalAgent`, and `ReactAgent`, each with a different prompting strategy (zero‑shot,

In [6]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='ae4af1c4-eac6-4949-bf1e-566092494154', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='2eaf7615-51b8-493e-84cb-2c80ce3f1dad', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='740a375c-e0b0-4dde-9c3a-4550a0f59e44', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order duri